# Named Entity Recognition

**AI3 · NLP Deep Dive** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~15 min

## Introduction

Up to now, we have mostly been concentrating on one NLP task - classification. However, there are also other NLP tasks that can be accomplished with neural networks. One of those tasks is **[Named Entity Recognition](https://wikipedia.org/wiki/Named-entity_recognition)** (NER), which deals with recognizing specific entities within text, such as places, person names, date-time intervals, chemical formulae and so on.



## Example of Using NER

Suppose you want to develop a natural language chat bot, similar to Amazon Alexa or Google Assistant. The way intelligent chat bots work is to *understand* what the user wants by doing text classification on the input sentence. The result of this classification is so-called **intent**, which determines what a chat bot should do.

![Bot NER](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/5-NLP/19-NER/images/bot-ner.png)

> Image by the author

However, a user may provide some parameters as part of the phrase. For example, when asking for the weather, she may specify a location or date. A bot should be able to understand those entities, and fill in the parameter slots accordingly before performing the action. This is exactly where NER comes in.

> ✅ Another example would be [analyzing scientific medical papers](https://soshnikov.com/science/analyzing-medical-papers-with-azure-and-text-analytics-for-health/). One of the main things we need to look for are specific medical terms, such as diseases and medical substances. While a small number of diseases can probably be extracted using substring search, more complex entities, such as chemical compounds and medication names, need a more complex approach.



## NER as Token Classification

NER models are essentially **token classification models**, because for each of the input tokens we need to decide whether it belongs to an entity or not, and if it does - to which entity class.

Consider the following paper title:

**Tricuspid valve regurgitation** and **lithium carbonate** **toxicity** in a newborn infant.

Entities here are:

* Tricuspid valve regurgitation is a disease (`DIS`)
* Lithium carbonate is a chemical substance (`CHEM`)
* Toxicity is also a disease (`DIS`)

Notice that one entity can span several tokens. And, as in this case, we need to distinguish between two consecutive entities. Thus, it is common to use two classes for each entity - one specifying the first token of the entity (often the `B-` prefix is used, for **b**eginning), and another - the continuation of an entity (`I-`, for **i**nner token). We also use `O` as a class to represent all **o**ther tokens. Such token tagging is called [BIO tagging](https://en.wikipedia.org/wiki/Inside%E2%80%93outside%E2%80%93beginning_(tagging)) (or IOB). When tagged, our title will look like this:

| Token | Tag
| ------|-----
| Tricuspid | B-DIS
| valve | I-DIS
| regurgitation | I-DIS
| and | O
| lithium | B-CHEM
| carbonate | I-CHEM
| toxicity | B-DIS
| in | O
| a | O
| newborn | O
| infant | O
| . | O

Since we need to build a one-to-one correspondence between tokens and classes, we can train a rightmost **many-to-many** neural network model from this picture:

![Image showing common recurrent neural network patterns.](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/5-NLP/17-GenerativeNetworks/images/unreasonable-effectiveness-of-rnn.jpg)

> *Image from [this blog post](http://karpathy.github.io/2015/05/21/rnn-effectiveness/) by [Andrej Karpathy](http://karpathy.github.io/). NER token classification models correspond to the right-most network architecture on this picture.*



## Training NER models

Since a NER model is essentially a token classification model, we can use RNNs that we are already familiar with for this task. In this case, each block of recurrent network will return the token ID. The following example notebook shows how to train LSTM for token classification.



## Example Notebooks: NER

Continue your learning in the following notebook:

* [NER with TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/19-NER/NER-TF.ipynb)



## Conclusion

A NER model is a **token classification model**, which means that it can be used to perform token classification. This is a very common task in NLP, helping to recognize specific entities within text including places, names, dates, and more.



## Challenge

Complete the assignment linked below to train a named entity recognition model for medical terms, then try it on a different dataset.



## Review & Self Study

Read through the blog [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/) and follow along with the Further Reading section in that article to deepen your knowledge.



## Assignment

In the assignment for this lesson, you will have to train a medical entity recognition model. You can start with training an LSTM model as described in this lesson, and proceed with using the BERT transformer model. Read [the instructions](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/19-NER/lab/README.md) to get all the details.



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

1. [BIO span extraction + span-level F1](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m03-span-f1)
2. [Majority-tag sequence baseline](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m03-majority-tagger)

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — BIO span extraction + span-level F1

●●● · ~15 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m03-span-f1)

> NER tasks (FAIO, NEOAI) are scored on whole entity spans, not per-tag accuracy. Convert the BIO tag list into a set of (start, end, type) spans — `end` exclusive — then score exact span matches with precision/recall/F1. A predicted span is correct only if its boundaries AND its type all match a gold span.

Implement `span_f1(gold, pred)` → the span-level F1 between two BIO tag lists (each a flat list of `'B-X'`/`'I-X'`/`'O'` strings). Span rules: `B-X` closes any open span and opens a new `(i, …, X)` span; `I-X` extends the open span only if its type is `X`, otherwise it opens a new `(i, …, X)` span itself; `O` closes the open span; a span left open at the end closes at `len(tags)`. Build a set of `(start, end, type)` per list, then `tp = |gold ∩ pred|`, `precision = tp/|pred|`, `recall = tp/|gold|`, `F1 = 2PR/(P+R)`. Return `1.0` when both sets are empty, `0.0` when exactly one is empty or `tp == 0` (no overlap → `P+R = 0`, guard the division).

```
gold=[B-PER,I-PER,O,B-LOC] pred=[B-PER,O,O,B-LOC]
→ gold spans {(0,2,PER),(3,4,LOC)}; pred {(0,1,PER),(3,4,LOC)}; tp=1 → P=1/2 R=1/2 → 0.5
```

<details><summary>Hint</summary>

Walk the list keeping open=(start,type); B/O/mismatched-I flush it into the set. Then f1 = 2*tp/(2*tp + fp + fn).

</details>


In [ ]:
import numpy as np

def span_f1(gold, pred):
    def spans(tags):
        # your code
        return set()
    g, p = spans(gold), spans(pred)
    # your code
    return 0.0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    types = [\"PER\", \"LOC\", \"ORG\"]\n    n = int(rng.integers(4, 12))\n    def make():\n        tags = []\n        while len(tags) < n:\n            if rng.random() < 0.55:\n                tags.append(\"O\")\n            else:\n                t = str(rng.choice(types))\n                L = int(rng.integers(1, 4))\n                tags.append(\"B-\" + t)\n                tags += [\"I-\" + t] * (L - 1)\n        return tags[:n]\n    gold = make()\n    if rng.random() < 0.15:\n        return {\"gold\": [\"O\"] * n, \"pred\": [\"O\"] * n}   # both empty -> 1.0\n    # corrupt a copy of gold so pred is usually close-but-not-equal\n    pred = list(gold)\n    for i in range(n):\n        if rng.random() < 0.3:\n            pred[i] = \"O\" if pred[i] != \"O\" else \"B-\" + str(rng.choice(types))\n        elif pred[i].startswith(\"I-\") and rng.random() < 0.25:\n            others = [t for t in types if t != pred[i][2:]]\n            pred[i] = \"I-\" + str(rng.choice(others))  # mismatched I- opens a new span\n    return {\"gold\": gold, \"pred\": pred}\n\ndef span_f1(gold, pred):\n    def spans(tags):\n        s, st, ty = set(), -1, None\n        for i, t in enumerate(tags):\n            if t.startswith(\"B-\"):\n                if st >= 0: s.add((st, i, ty))\n                st, ty = i, t[2:]\n            elif t.startswith(\"I-\"):\n                if st < 0 or ty != t[2:]:\n                    if st >= 0: s.add((st, i, ty))\n                    st, ty = i, t[2:]\n            else:\n                if st >= 0: s.add((st, i, ty))\n                st, ty = -1, None\n        if st >= 0: s.add((st, len(tags), ty))\n        return s\n    g, p = spans(gold), spans(pred)\n    tp = len(g & p)\n    if not g and not p: return 1.0\n    if tp == 0: return 0.0\n    P, R = tp / len(p), tp / len(g)\n    return 2 * P * R / (P + R)", "span_f1", 20)

### Task 2 — Majority-tag sequence baseline

●●○ · ~30 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m03-majority-tagger)

> Before any real tagger, build the majority baseline: each token gets the tag it most often had in training data, and unseen tokens get the globally most frequent tag. On NER/POS data this baseline is often fairly strong, and it shows how much a real model has to add.

Implement `majority_tagger(train_tokens, train_tags, test_tokens)` → list of predicted tags for `test_tokens`. Use each token's most frequent tag in the parallel training lists; ties go to the alphabetically smaller tag. Tokens never seen in training get the globally most frequent tag (same tie rule).

<details><summary>Hint</summary>

A dict of collections.Counter per token, plus one global Counter for the fallback; resolve each with min(counter, key=lambda t: (-count, t)).

</details>


In [ ]:
import numpy as np

def majority_tagger(train_tokens, train_tags, test_tokens):
    # your code
    return [train_tags[0][0] for _ in test_tokens]

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    words = [\"cat\", \"run\", \"quick\", \"bob\", \"jump\", \"red\"]\n    tags = [\"N\", \"V\", \"A\"]\n    tt, tg = [], []\n    for _ in range(int(rng.integers(3, 8))):\n        L = int(rng.integers(2, 6))\n        ws = [str(w) for w in rng.choice(words, L)]\n        tt.append(ws)\n        tg.append([str(t) for t in rng.choice(tags, L)])\n    test = [str(w) for w in rng.choice(words, int(rng.integers(2, 6)))]\n    return {\"train_tokens\": tt, \"train_tags\": tg, \"test_tokens\": test}\n\ndef majority_tagger(train_tokens, train_tags, test_tokens):\n    per = {}\n    glob = collections.Counter()\n    for ws, ts in zip(train_tokens, train_tags):\n        for w, t in zip(ws, ts):\n            per.setdefault(w, collections.Counter())[t] += 1\n            glob[t] += 1\n    best = {w: min(c, key=lambda t: (-c[t], t)) for w, c in per.items()}\n    default = min(glob, key=lambda t: (-glob[t], t))\n    return [best.get(w, default) for w in test_tokens]", "majority_tagger", 20)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/19-NER)